In [17]:
from pathlib import Path
import pandas as pd

base_dir = Path(
    r"D:\NTU_PSY\Documents\Patric_Asen_BrainHackProject\subject_fMRI_nii"
)

all_subjects = ["sub-01", "sub-02", "sub-03"]

all_subjects_clean = []

for sub in all_subjects:

    print(f"\n===== Processing {sub} =====")

    sub_dir = base_dir / sub
    session_dirs = sorted(sub_dir.glob("ses-things*"))

    all_conditions = []

    for ses_dir in session_dirs:

        ses_name = ses_dir.name

        for run_idx in range(1, 11):

            tsv_filename = f"{sub}_{ses_name}_run-{run_idx:02d}_conditions.tsv"
            tsv_path = ses_dir / tsv_filename

            if not tsv_path.exists():
                print(f"Missing: {tsv_path}")
                continue

            df = pd.read_csv(tsv_path, sep="\t")

            df["session"] = ses_name
            df["run"] = run_idx
            df["trial_idx"] = df.index
            df["subject"] = sub

            all_conditions.append(df)

    if len(all_conditions) == 0:
        print(f"No data found for {sub}")
        continue

    df_sub = pd.concat(all_conditions, ignore_index=True)

    # -------------------------
    # 清理 Unnamed column
    # -------------------------
    df_sub = df_sub.drop(columns=["Unnamed: 0"], errors="ignore")

    print("Before removing catch:", len(df_sub))

    # -------------------------
    # catch trial removal
    # -------------------------
    df_sub = df_sub[
        df_sub["image_filename"].str.contains("/", na=False)
    ].copy()

    df_sub["concept"] = (
        df_sub["image_filename"]
        .str.split("/")
        .str[0]
    )

    print("After removing catch:", len(df_sub))

    # -------------------------
    # repeated image removal
    # -------------------------
    img_count = (
        df_sub
        .groupby(["concept", "image_filename"])
        .size()
        .reset_index(name="count")
    )

    repeated_imgs = img_count[img_count["count"] == 12]
    repeated_imgs_list = repeated_imgs["image_filename"]

    df_sub_clean = df_sub[
        ~df_sub["image_filename"].isin(repeated_imgs_list)
    ].copy()

    print("After removing repeated images:", len(df_sub_clean))
    print("concepts:", df_sub_clean["concept"].nunique())

    # -------------------------
    # save per subject
    # -------------------------
    out_path = base_dir / f"{sub}_condition_clean.csv"
    df_sub_clean.to_csv(out_path, index=False)

    all_subjects_clean.append(df_sub_clean)

# -------------------------
# merge all subjects
# -------------------------
df_all = pd.concat(all_subjects_clean, ignore_index=True)

print("\nDONE")
print(df_all.shape)


===== Processing sub-01 =====
Before removing catch: 11040
After removing catch: 9840
After removing repeated images: 8640
concepts: 720

===== Processing sub-02 =====
Before removing catch: 11040
After removing catch: 9840
After removing repeated images: 8640
concepts: 720

===== Processing sub-03 =====
Before removing catch: 11040
After removing catch: 9840
After removing repeated images: 8640
concepts: 720

DONE
(25920, 6)
